# DeepSWE eval on Agent Sandbox warm pools (no model)

This notebook runs the **R2E-Gym** SWE environment — the very same `RepoEnv` that
[tunix](https://github.com/google/tunix) deepswe drives — on sandboxes
**pre-warmed** by an `agent-sandbox-rl` `SandboxFleet`, instead of cold-creating a
sandbox per task.

```
tunix SWEEnv  →  R2E-Gym RepoEnv(backend="kubernetes-sandbox")  →  DockerRuntime  →  a sandbox
```

R2E-Gym's `kubernetes-sandbox` backend normally cold-creates a sandbox per env
(and `eval_deepswe.py` re-implements warm pools inline). The
`agent_sandbox_rl.adapters.r2egym` adapter instead **binds a fleet-warmed pod**
into R2E-Gym's `RepoEnv`, so the same env/reward path reuses the package's
v1beta1, concurrency-sized, observed warm pools.

**No model / TPU required.** A *stub policy* exercises the env path so you can
develop, benchmark, and tune the infrastructure (warm pools, sizing, claim
latency, observability) on its own. A real rollout swaps the stub for
`env.step(model_action)` and, optionally, `env.compute_reward()`.

The notebook runs in one of two tiers, chosen automatically:

| Tier | When | What each task does |
| :--- | :--- | :--- |
| **B** | R2E-Gym importable | Build a `FleetRepoEnv` on a warm pod → read the task instruction → run one stub action via the real R2E-Gym runtime. The full integration path. |
| **A** | R2E-Gym absent | Fall back to a router-free `handle.exec` probe. The fleet/warm-pool path is identical; only the in-sandbox work differs. |

## Prerequisites

**Cluster**
- A Kubernetes cluster with the **Agent Sandbox controller + v1beta1 extensions**
  installed (the `SandboxTemplate` / `SandboxWarmPool` / `SandboxClaim` /
  `Sandbox` CRDs). `fleet.preflight()` checks this and raises `PreflightError`
  with a precise message if something is missing.
- A `kubectl` context pointing at it. **On GKE** also install the
  `gke-gcloud-auth-plugin` (`gcloud components install gke-gcloud-auth-plugin`).
- Worker capacity matching the `NODE_SELECTOR_*` you set below (and a matching
  `runtime_class` if you require one, e.g. gVisor).

**Python packages** (from the repo root)
```bash
pip install -e clients/python/agentic-sandbox-client \
            -e 'examples/agent-sandbox-rl[swebench]'
```

**For Tier B only — R2E-Gym** (optional)
```bash
pip install -e path/to/R2E-Gym          # the sibling checkout
```
> ⚠️ R2E-Gym imports the legacy `gym` package, which has **no installable
> distribution on recent Python (3.12+)**. On such interpreters the import fails
> and the notebook automatically runs **Tier A**. For Tier B use an environment
> where `gym` (or a `gym`-compatible shim) resolves.

**Notes**
- SWE-bench task images are multi-GB. Keep `TASKS_LIMIT` small and allow time for
  the first pull (raise `SANDBOX_READY_TIMEOUT`); subsequent pulls reuse cached,
  layer-shared base images.
- Everything created is labeled `app=agent-sandbox-rl` and torn down at the end —
  see the final cell.

## 1. Imports

In [ ]:
# Copyright 2026 The Kubernetes Authors.
# Licensed under the Apache License, Version 2.0 (the "License").
# SPDX-License-Identifier: Apache-2.0
import json
import logging
import os

from agent_sandbox_rl import (
    ClusterConfig,
    FleetConfig,
    SandboxFleet,
    TemplateSpec,
)
from agent_sandbox_rl.adapters.swebench import SWEBENCH_PROBE, SweBenchSource

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s %(levelname)s %(name)s %(message)s")

## 2. Configuration

All knobs are environment-overridable so the notebook doubles as a script.

| Variable | Meaning |
| :--- | :--- |
| `NAMESPACE` | Namespace for the run's CRs/pods. |
| `TASKS_LIMIT` | How many SWE-bench rows to run (keep small for a demo). |
| `WARMPOOL_STRATEGY` | `none` / `naive` / `sliding` — *when* pools are warm. |
| `MAX_CONCURRENT` | Parallel claim+exec; also the pool-sizing budget. |
| `MAX_WARMPOOL_SIZE` | Hard cap on replicas per image pool. |
| `SANDBOX_READY_TIMEOUT` | Seconds to wait for a pool to be ready (cold pull). |
| `NODE_SELECTOR_KEY/VAL` | Pin sandboxes to a node pool. |
| `RUN_REWARD` | Tier B: run the real test suite via `compute_reward()` (slow). |
| `REPORT_DIR` | If set, persist the RunReport there. |

In [ ]:
NAMESPACE = os.getenv("NAMESPACE", "rl-tunix-swebench")
TASKS_LIMIT = int(os.getenv("TASKS_LIMIT", "5"))
STRATEGY = os.getenv("WARMPOOL_STRATEGY", "sliding")
MAX_CONCURRENT = int(os.getenv("MAX_CONCURRENT", "5"))
MAX_WARMPOOL_SIZE = int(os.getenv("MAX_WARMPOOL_SIZE", "8"))
READY_TIMEOUT = int(os.getenv("SANDBOX_READY_TIMEOUT", "1200"))
RUN_REWARD = os.getenv("RUN_REWARD", "0") == "1"   # Tier B: real tests, slow
REPORT_DIR = os.getenv("REPORT_DIR", "")

node_selector = None
if os.getenv("NODE_SELECTOR_KEY") and os.getenv("NODE_SELECTOR_VAL"):
    node_selector = {os.environ["NODE_SELECTOR_KEY"]: os.environ["NODE_SELECTOR_VAL"]}

print(f"namespace={NAMESPACE} strategy={STRATEGY} tasks={TASKS_LIMIT} "
      f"concurrency={MAX_CONCURRENT} node_selector={node_selector}")

## 3. Build the fleet and load SWE-bench tasks

`SweBenchSource(keep_row=True)` is **required** for the R2E-Gym path: it stores
the full dataset row under `task.metadata["ds"]`, which R2E-Gym's env and reward
grading need. (Without it the adapter raises a clear error; Tier A still works.)

In [ ]:
fleet = SandboxFleet(FleetConfig(
    clusters=[ClusterConfig(name="default", namespace=NAMESPACE)],
    max_concurrent=MAX_CONCURRENT,
    max_warmpool_size=MAX_WARMPOOL_SIZE,
    ready_timeout=READY_TIMEOUT,
    template=TemplateSpec(node_selector=node_selector),
))
fleet.load_tasks(SweBenchSource(limit=TASKS_LIMIT, keep_row=True))
print(f"loaded {len(fleet.tasks)} tasks across {len(fleet.image_counts())} images")

## 4. Per-task work: Tier B (R2E-Gym) or Tier A (exec probe)

We try to import the R2E-Gym adapter. If R2E-Gym is installed we use the real
`FleetRepoEnv` (Tier B); otherwise we fall back to a router-free probe (Tier A)
and print a banner — the notebook never hard-fails.

The `rollout(task, handle)` function is the **stub policy**. In Tier B it builds
an env bound to the warm pod, reads the task instruction, and runs one trivial
action through R2E-Gym's real runtime. A production rollout would instead loop
`env.step(model_action)` until done and call `env.compute_reward()`.

**Ownership:** `env.close()` is a no-op that does *not* delete the pod — the fleet
owns the pod's lifecycle and releases it via `fleet.run` / `fleet.release`.

In [ ]:
try:
    from agent_sandbox_rl.adapters.r2egym import (
        make_fleet_repo_env,
        r2egym_command_files,
    )
    _ = r2egym_command_files()          # forces the r2egym import; raises if absent
    HAVE_R2EGYM = True
    print("[Tier B] R2E-Gym available — using the real RepoEnv on warm pods.")
except Exception as e:  # noqa: BLE001
    HAVE_R2EGYM = False
    print(f"[Tier A] R2E-Gym not available ({e}); using exec-only probe.")


def rollout(task, handle):
    """Stub policy. Real rollouts replace the body with env.step(model_action)."""
    if not HAVE_R2EGYM:
        out = handle.exec(SWEBENCH_PROBE).strip()
        return {"id": task.id, "tier": "A", "probe": out.splitlines()[:1]}

    env = make_fleet_repo_env(handle, command_files=r2egym_command_files(),
                              verbose=False)
    try:
        instruction = env.get_task_instruction()
        # Stub action via R2E-Gym's real runtime exec (namespace-correct, bound
        # to the warm pod). A policy would instead call env.step(action).
        listing, _code = env.runtime.run("ls /testbed | head -n 5")
        result = {"id": task.id, "tier": "B",
                  "instruction_chars": len(instruction or ""),
                  "testbed_head": listing.strip().splitlines()[:5]}
        if RUN_REWARD:
            result["reward"] = env.compute_reward()   # real test suite (slow)
        return result
    finally:
        env.close()        # no-op teardown; the fleet releases the pod

## 5. Run all tasks under the warm-pool strategy

`fleet.run(...)` provisions pools per the strategy, claims a warm sandbox per
task, runs `rollout` (up to `MAX_CONCURRENT` in parallel), releases each, and
tears everything down — returning one result per task and populating
`fleet.report`.

In [ ]:
try:
    results = fleet.run(rollout, strategy=STRATEGY, concurrency=MAX_CONCURRENT)
    print("\n=== results ===")
    for r in results:
        print(" ", r)
finally:
    # fleet.run already tears down; belt-and-suspenders for a partial/aborted run.
    if fleet.plan_ is not None:
        fleet.teardown()

## 6. The RunReport

`fleet.report` is the per-run performance summary: an `environment` block
(cluster context / k8s version / nodes / node pools / region) plus per-phase
timings (`preflight`, `plan`, `create_warmpool`, `wait_pool_ready`, `claim`,
`process`, `release`, `teardown`) and the claim / task / warm-replica counters.

See [`../performance_reports/README.md`](../performance_reports/README.md) for a
full breakdown of every phase and metric (and the note that summed phase totals
exceed the wall-clock `TOTAL` under concurrency).

In [ ]:
print(fleet.report.summary())

## 7. Persist the report (optional)

If `REPORT_DIR` is set, write a timestamped `.txt` (the table) and `.json`
(`report.to_dict()`), matching the convention used by `run_swebench_fleet.py`.

In [ ]:
if REPORT_DIR and fleet.report is not None:
    import datetime
    import pathlib

    out = pathlib.Path(REPORT_DIR)
    out.mkdir(parents=True, exist_ok=True)
    stamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    base = out / f"deepswe_{STRATEGY}_{len(fleet.tasks)}tasks_{stamp}"
    base.with_suffix(".txt").write_text(fleet.report.summary() + "\n")
    base.with_suffix(".json").write_text(
        json.dumps(fleet.report.to_dict(), indent=2) + "\n")
    print(f"wrote {base}.txt / .json")
else:
    print("REPORT_DIR not set — skipping report file (report still in fleet.report)")

## 8. Cleanup

`fleet.run` already tears down everything it created (claims → pools → templates,
by the `app=agent-sandbox-rl` label). If you instead drove the primitives
(`fleet.setup()` / `acquire` / `release`) yourself, call `fleet.teardown()` when
done. To double-check the namespace is clean:

```bash
kubectl get sandboxwarmpools,sandboxtemplates,sandboxclaims,sandboxes \
  -n "$NAMESPACE"
```